# Linear and logistic regression from scratch (numpy only)

This notebook goes with `notes.md` §1 (linear regression), §2 (logistic
regression) and §3 (regularization). It fits linear regression twice, once
with the normal equation and once with gradient descent, and checks that both
agree with each other and with sklearn. It then shows ridge shrinkage, and
trains logistic regression by gradient descent with and without an L2 penalty.

All randomness comes from one fixed seed, and the whole notebook runs in a few
seconds on a CPU.

In [1]:
import numpy as np

np.random.seed(42)

## 1. Linear regression with the normal equation (notes §1.2)

We generate 500 examples with 5 features from $y = Xw + b + \varepsilon$, with
true weights $w = (2, -1, 0.5, 0, 3)$, bias $b = 1.5$ and noise standard
deviation 0.5. A column of ones is prepended so the bias is just another
weight. The normal equation $X^\top X w = X^\top y$ is solved with
`np.linalg.solve` (no explicit inverse). Expect the fitted weights to be close
to $(1.5, 2, -1, 0.5, 0, 3)$ (bias first) and the training MSE to be close to
the noise variance $0.5^2 = 0.25$.

In [2]:
def add_bias(X):
    # Prepend a column of ones so the bias is just another weight.
    return np.hstack([np.ones((X.shape[0], 1)), X])


def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))


def linreg_normal_equation(X, y, l2=0.0):
    # Solve (X^T X + l2*I) w = X^T y. With l2 > 0 this is ridge regression;
    # the bias (first weight) is not penalized.
    Xb = add_bias(X)
    d = Xb.shape[1]
    reg = l2 * np.eye(d)
    reg[0, 0] = 0.0
    return np.linalg.solve(Xb.T @ Xb + reg, Xb.T @ y)


n, d = 500, 5
X = np.random.randn(n, d)
true_w = np.array([2.0, -1.0, 0.5, 0.0, 3.0])
true_b = 1.5
y = X @ true_w + true_b + 0.5 * np.random.randn(n)

w_ne = linreg_normal_equation(X, y)
print(f"normal-equation weights (bias first): {np.round(w_ne, 4)}")
print(f"training MSE = {mse(y, add_bias(X) @ w_ne):.6f}")

normal-equation weights (bias first): [ 1.5113  2.0056 -1.0046  0.4789 -0.0246  3.0092]
training MSE = 0.252171


## 2. The same fit by gradient descent (notes §1.3)

Gradient descent on the mean squared error uses the gradient
$\frac{2}{n} X^\top (Xw - y)$: for each weight, the residual times the feature
value, averaged over examples. The features here are standard normal, so the
problem is well conditioned and a learning rate of 0.1 converges quickly. The
printed MSE should fall to the normal-equation value, and the assert checks
that the two weight vectors agree to within $10^{-6}$.

In [3]:
def linreg_gradient_descent(X, y, lr=0.1, n_iters=5000, l2=0.0, verbose_every=1000):
    # Minimize (1/n)||Xw - y||^2 + (l2/n)||w_{1:}||^2 by batch gradient descent.
    Xb = add_bias(X)
    n, d = Xb.shape
    w = np.zeros(d)
    for it in range(n_iters):
        resid = Xb @ w - y
        grad = (2.0 / n) * (Xb.T @ resid)
        grad[1:] += 2.0 * l2 * w[1:] / n   # same scaling as the normal equation
        w -= lr * grad
        if verbose_every and (it % verbose_every == 0 or it == n_iters - 1):
            print(f"    iter {it:5d}  MSE = {mse(y, Xb @ w):.6f}")
    return w


w_gd = linreg_gradient_descent(X, y, lr=0.1, n_iters=5000)
print(f"GD weights (bias first): {np.round(w_gd, 4)}")

diff = float(np.max(np.abs(w_ne - w_gd)))
print(f"max |normal equation - GD| over weights: {diff:.2e}")
assert diff < 1e-6, "normal equation and gradient descent disagree"
print("PASS: normal equation and gradient descent agree.")

    iter     0  MSE = 11.027592
    iter  1000  MSE = 0.252171
    iter  2000  MSE = 0.252171
    iter  3000  MSE = 0.252171
    iter  4000  MSE = 0.252171
    iter  4999  MSE = 0.252171
GD weights (bias first): [ 1.5113  2.0056 -1.0046  0.4789 -0.0246  3.0092]
max |normal equation - GD| over weights: 1.78e-15
PASS: normal equation and gradient descent agree.


## 3. Check against sklearn

`sklearn.linear_model.LinearRegression` solves the same least-squares problem.
Its intercept and coefficients should match ours to about $10^{-8}$.

In [4]:
from sklearn.linear_model import LinearRegression

sk = LinearRegression().fit(X, y)
sk_w = np.concatenate([[sk.intercept_], sk.coef_])
sk_diff = float(np.max(np.abs(w_ne - sk_w)))
print(f"max |ours - sklearn| = {sk_diff:.2e}")
assert sk_diff < 1e-8, "disagrees with sklearn"
print("PASS: matches sklearn LinearRegression.")

max |ours - sklearn| = 1.78e-15
PASS: matches sklearn LinearRegression.


## 4. Ridge shrinks the weights (notes §3.1)

Ridge adds $\lambda \|w\|^2$ to the squared error, which changes the normal
equation to $(X^\top X + \lambda I) w = X^\top y$. The bias is left
unpenalized (notes §3.3, trap 1). With $\lambda = 100$ on 500 examples, the
norm of the feature weights should come out clearly smaller than the
unregularized norm.

In [5]:
w_ridge = linreg_normal_equation(X, y, l2=100.0)
print(f"ridge (l2=100) weights: {np.round(w_ridge, 4)}")
print(f"||w||_2 unregularized = {np.linalg.norm(w_ne[1:]):.4f}, "
      f"ridge = {np.linalg.norm(w_ridge[1:]):.4f}")
assert np.linalg.norm(w_ridge[1:]) < np.linalg.norm(w_ne[1:])
print("PASS: ridge weights have a smaller norm.")

ridge (l2=100) weights: [ 1.5769  1.6431 -0.8291  0.418  -0.0218  2.4932]
||w||_2 unregularized = 3.7837, ridge = 3.1271
PASS: ridge weights have a smaller norm.


## 5. Logistic regression by gradient descent (notes §2.3)

Two Gaussian blobs (400 points each, centers $(-2, -1)$ and $(1.5, 1)$) give a
2-class problem that is close to linearly separable; 600 points are used for
training and 200 for testing. The gradient of the mean cross-entropy is
$\frac{1}{n} X^\top (p - y)$, the same residual-times-input form as linear
regression, with no sigmoid-derivative factor. The sigmoid is computed in a
form that does not overflow for large $|z|$. Expect the cross-entropy to
decrease steadily and test accuracy to be around 98%.

In [6]:
def sigmoid(z):
    # Numerically stable: never exponentiates a large positive number.
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out


def cross_entropy(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def logreg_gradient_descent(X, y, lr=0.5, n_iters=3000, l2=0.0, verbose_every=500):
    # Minimize mean cross-entropy + (l2/n)*||w_{1:}||^2.
    Xb = add_bias(X)
    n, d = Xb.shape
    w = np.zeros(d)
    for it in range(n_iters):
        p = sigmoid(Xb @ w)
        grad = (Xb.T @ (p - y)) / n          # the (p - y) x gradient
        grad[1:] += 2.0 * l2 * w[1:] / n
        w -= lr * grad
        if verbose_every and (it % verbose_every == 0 or it == n_iters - 1):
            print(f"    iter {it:5d}  CE = {cross_entropy(y, p):.6f}")
    return w


def logreg_predict_proba(X, w):
    return sigmoid(add_bias(X) @ w)


n_per = 400
X0 = np.random.randn(n_per, 2) + np.array([-2.0, -1.0])
X1 = np.random.randn(n_per, 2) + np.array([1.5, 1.0])
Xc = np.vstack([X0, X1])
yc = np.concatenate([np.zeros(n_per), np.ones(n_per)])
perm = np.random.permutation(len(yc))
Xc, yc = Xc[perm], yc[perm]

n_train = 600
Xtr, ytr = Xc[:n_train], yc[:n_train]
Xte, yte = Xc[n_train:], yc[n_train:]

w = logreg_gradient_descent(Xtr, ytr, lr=0.5, n_iters=3000)
p_tr = logreg_predict_proba(Xtr, w)
p_te = logreg_predict_proba(Xte, w)
acc_tr = float(np.mean((p_tr > 0.5) == ytr))
acc_te = float(np.mean((p_te > 0.5) == yte))
print(f"weights (bias first): {np.round(w, 4)}")
print(f"train CE = {cross_entropy(ytr, p_tr):.4f}, acc = {acc_tr:.4f}")
print(f"test  CE = {cross_entropy(yte, p_te):.4f}, acc = {acc_te:.4f}")

    iter     0  CE = 0.693147
    iter   500  CE = 0.074401
    iter  1000  CE = 0.074331
    iter  1500  CE = 0.074330
    iter  2000  CE = 0.074330
    iter  2500  CE = 0.074330
    iter  2999  CE = 0.074330
weights (bias first): [0.8432 3.1374 1.5271]
train CE = 0.0743, acc = 0.9683
test  CE = 0.0587, acc = 0.9850


## 6. Logistic regression with an L2 penalty (notes §3)

The same training run with an L2 penalty ($l2 = 1$) on the feature weights.
The penalty pulls the weights toward zero, so their norm should drop, while
test accuracy should stay about the same because the classes are well
separated.

In [7]:
w_l2 = logreg_gradient_descent(Xtr, ytr, lr=0.5, n_iters=3000, l2=1.0,
                               verbose_every=0)
p_te_l2 = logreg_predict_proba(Xte, w_l2)
acc_te_l2 = float(np.mean((p_te_l2 > 0.5) == yte))
print(f"weights: {np.round(w_l2, 4)}")
print(f"||w|| unregularized = {np.linalg.norm(w[1:]):.4f}, "
      f"L2-regularized = {np.linalg.norm(w_l2[1:]):.4f}")
print(f"test acc = {acc_te_l2:.4f}")

weights: [0.6657 2.4955 1.2465]
||w|| unregularized = 3.4893, L2-regularized = 2.7896
test acc = 0.9850


## 7. Check against sklearn

sklearn's `LogisticRegression` always applies some L2 penalty, controlled by
$C$ (inverse regularization strength). Setting $C = 10^6$ makes it nearly
unregularized, so its test accuracy should be within 2 percentage points of
our unregularized model.

In [8]:
from sklearn.linear_model import LogisticRegression

sk = LogisticRegression(C=1e6, max_iter=5000).fit(Xtr, ytr)
sk_acc = float(sk.score(Xte, yte))
print(f"sklearn LogisticRegression test acc = {sk_acc:.4f} (ours: {acc_te:.4f})")
assert abs(sk_acc - acc_te) <= 0.02, "accuracy differs from sklearn by more than 2%"
print("PASS: within 2% of sklearn's test accuracy.")

sklearn LogisticRegression test acc = 0.9850 (ours: 0.9850)
PASS: within 2% of sklearn's test accuracy.
